# Chapter 11: Neural Networks that Understand Language — *King − Man + Woman ≈ Queen*

**Tujuan pembelajaran:**
1. Memahami NLP dan klasifikasi sentimen sebagai *supervised learning*.
2. Mengubah teks ke representasi *bag of words* dan *word embedding*.
3. Mengimplementasikan jaringan sederhana untuk klasifikasi sentimen dengan NumPy.
4. Membandingkan kata berdasarkan jarak vektor embedding.
5. Membuat model *fill in the blank* dengan *negative sampling*.
6. Memahami mengapa **target dan loss function** menentukan representasi yang dipelajari.
7. Menguji ide *word analogy* seperti `king - man + woman ≈ queen` secara kritis.

## 1. Apa arti “memahami bahasa” bagi neural network?

Dalam konteks bab ini, jaringan saraf tidak memahami kalimat seperti manusia. Jaringan saraf **mempelajari keteraturan statistik** yang membantu memprediksi suatu target.

Contoh tugas NLP:
- **Sentiment analysis**: menentukan apakah ulasan bersifat positif atau negatif.
- **Text classification**: mengklasifikasikan topik dokumen.
- **Language modeling**: memperkirakan kata yang mungkin muncul berdasarkan konteks.
- **Word representation**: memetakan kata ke vektor numerik (*word embeddings*).

Pertanyaan kunci bab ini: **informasi apa yang bisa dipelajari dari bahasa jika kita mengubah jenis prediksinya?**

> Penting: kemampuan menghasilkan prediksi yang benar **bukan bukti** bahwa model memiliki pemahaman semantik menyeluruh. Interpretasi ini juga ditekankan saat buku membahas keterbatasan representasi berdasarkan kosa kata (hlm. 196–200).

## 2. Supervised NLP dan dataset IMDB

Pada *supervised learning*, data pelatihan terdiri atas pasangan `(input, target)`.

Untuk klasifikasi sentimen:

`Teks ulasan → Representasi numerik → Neural network → Probabilitas ulasan positif`

Buku menggunakan ulasan IMDB dengan dua label, **positive** dan **negative** (hlm. 191–196). *Label* dikodekan menjadi `1` untuk positif dan `0` untuk negatif. Contoh mini dalam notebook ini menggunakan **bahasa Inggris** agar kosakata lebih dekat dengan korpus IMDB asli, tetapi seluruh penjelasan tetap berbahasa Indonesia.

**Batasan eksperimen:** Jika `reviews.txt`/`labels.txt` tidak tersedia, dipakai dataset kecil buatan sendiri hanya untuk mendemonstrasikan algoritma. Data buatan bukan sampel acak IMDB dan tidak cocok untuk kesimpulan performa umum.

In [1]:
import numpy as np
import re
from pathlib import Path
from collections import Counter

SEED = 42
rng = np.random.default_rng(SEED)

# Aktifkan hanya setelah menambahkan reviews.txt dan labels.txt ke folder notebook.
USE_IMDB_FILES = False
MAX_IMDB_REVIEWS = 2000  # batas awal agar percobaan CPU tetap praktis

ulasan_positif = [
    "this film was excellent and inspiring", "the story was beautiful and moving",
    "i loved the brilliant acting", "a wonderful movie with a great ending",
    "the actors gave amazing performances", "this film is entertaining and funny",
    "what a fantastic and thoughtful story", "a charming movie with lovely characters",
    "the plot was interesting and clever", "the visuals were beautiful and stunning",
    "an excellent film with wonderful music", "this was a delightful cinematic experience",
    "the movie felt warm and inspiring", "great direction and brilliant writing",
    "i enjoyed every minute of this movie", "a beautiful and emotionally rich film",
    "wonderful acting and an engaging plot", "the ending was satisfying and touching",
    "a smart funny and enjoyable movie", "excellent performances from the whole cast",
    "the film was creative and memorable", "a fantastic story told with heart",
    "the soundtrack was lovely and moving", "a brilliant movie with great characters",
    "the comedy was sharp and enjoyable", "a powerful and beautiful story",
    "i recommend this wonderful film", "the cast was excellent and convincing",
    "this movie is truly entertaining", "a lovely film with a clever plot",
    "the writing was strong and inspiring", "an amazing movie that made me smile",
]
ulasan_negatif = [
    "this film was terrible and disappointing", "the story was boring and weak",
    "i hated the awful acting", "a horrible movie with a bad ending",
    "the actors gave poor performances", "this film is dull and annoying",
    "what a dreadful and confusing story", "a tedious movie with irritating characters",
    "the plot was predictable and silly", "the visuals were ugly and cheap",
    "a terrible film with awful music", "this was a frustrating cinematic experience",
    "the movie felt empty and boring", "poor direction and weak writing",
    "i disliked every minute of this movie", "a bad and painfully slow film",
    "awful acting and a confusing plot", "the ending was disappointing and pointless",
    "a messy boring and unpleasant movie", "terrible performances from the whole cast",
    "the film was lazy and forgettable", "a horrible story told without care",
    "the soundtrack was noisy and irritating", "a dreadful movie with weak characters",
    "the comedy was forced and annoying", "a shallow and terrible story",
    "i cannot recommend this awful film", "the cast was poor and unconvincing",
    "this movie is truly boring", "a bad film with a silly plot",
    "the writing was weak and confusing", "an awful movie that made me angry",
]

if USE_IMDB_FILES:
    jalur_ulasan, jalur_label = Path('reviews.txt'), Path('labels.txt')
    if not (jalur_ulasan.exists() and jalur_label.exists()):
        raise FileNotFoundError('Simpan reviews.txt dan labels.txt di folder kerja, lalu jalankan ulang.')
    semua_ulasan = jalur_ulasan.read_text(encoding='utf-8').splitlines()
    label_teks = jalur_label.read_text(encoding='utf-8').splitlines()
    if len(semua_ulasan) != len(label_teks):
        raise ValueError('Jumlah ulasan dan label harus sama.')
    semua_label = np.array([1 if x.strip().lower() == 'positive' else 0
                            for x in label_teks], dtype=np.int64)
    # Subsampling acak per kelas untuk menghindari mengambil satu kelas saja.
    idx0 = rng.permutation(np.flatnonzero(semua_label == 0))[:MAX_IMDB_REVIEWS // 2]
    idx1 = rng.permutation(np.flatnonzero(semua_label == 1))[:MAX_IMDB_REVIEWS // 2]
    idx = rng.permutation(np.concatenate([idx0, idx1]))
    ulasan = [semua_ulasan[i] for i in idx]
    label = semua_label[idx]
    sumber_data = 'IMDB (subset lokal)'
else:
    ulasan = ulasan_positif + ulasan_negatif
    label = np.array([1] * len(ulasan_positif) + [0] * len(ulasan_negatif))
    sumber_data = 'Korpus demonstrasi (ditulis untuk notebook ini)'

print('Sumber data:', sumber_data)
print('Jumlah ulasan:', len(ulasan))
print('Jumlah positif:', int(sum(label)), '| negatif:', int(len(label) - sum(label)))
print('Contoh:', ulasan[0], '=>', label[0])

Sumber data: Korpus demonstrasi (ditulis untuk notebook ini)
Jumlah ulasan: 64
Jumlah positif: 32 | negatif: 32
Contoh: this film was excellent and inspiring => 1


## 3. Tokenisasi dan pembuatan vocabulary

Sebelum digunakan oleh neural network, teks diubah menjadi kumpulan token (kata). **Tokenisasi** merupakan operasi pemisahan teks; contohnya `"the movie was great"` menjadi `['the', 'movie', 'was', 'great']`.

Setiap kata kemudian diberi nomor indeks melalui `word2index`.

**Pilihan implementasi:** Buku menggunakan pemisahan berbasis spasi dan `set` untuk *bag of words* (hlm. 192–193). Di sini digunakan regex sederhana untuk menormalkan huruf besar dan tanda baca. Normalisasi ini merupakan adaptasi, bukan salinan persis prapemrosesan buku.

Untuk evaluasi yang benar, vocabulary dibentuk **hanya dari data latih** agar data uji tidak memengaruhi proses pembelajaran. Data dibagi acak secara terstratifikasi berdasarkan kelas.

In [2]:
def tokenisasi(teks):
    return re.findall(r"[a-z]+(?:'[a-z]+)?", teks.lower())

# Pembagian data stratified secara deterministik: 80% latih, 20% uji.
idx_latih, idx_uji = [], []
for kelas in (0, 1):
    indeks = rng.permutation(np.flatnonzero(label == kelas))
    n_uji = max(1, int(np.ceil(0.2 * len(indeks))))
    idx_uji.extend(indeks[:n_uji])
    idx_latih.extend(indeks[n_uji:])
idx_latih = rng.permutation(idx_latih)
idx_uji = rng.permutation(idx_uji)

teks_latih = [ulasan[i] for i in idx_latih]
teks_uji = [ulasan[i] for i in idx_uji]
y_latih, y_uji = label[idx_latih], label[idx_uji]

vocab = sorted({kata for teks in teks_latih for kata in tokenisasi(teks)})
word2index = {kata: i for i, kata in enumerate(vocab)}
index2word = {i: kata for kata, i in word2index.items()}

print('Vocabulary:', len(vocab), 'kata')
print('10 kata pertama:', vocab[:10])
print('Latih:', len(teks_latih), '| uji:', len(teks_uji))
assert len(set(idx_latih) & set(idx_uji)) == 0

Vocabulary: 102 kata
10 kata pertama: ['a', 'acting', 'actors', 'amazing', 'an', 'and', 'annoying', 'awful', 'bad', 'beautiful']
Latih: 50 | uji: 14


## 4. One-hot encoding dan bag of words

**One-hot vector** untuk satu kata memiliki satu elemen bernilai 1 pada posisi kata tersebut, dan nol di semua posisi lain.

Dalam **bag of words** (BoW), vektor sebuah dokumen menyatakan kata yang muncul dalam dokumen tanpa memperhatikan urutannya. Buku menggunakan representasi biner: kemunculan kata `1` atau tidak muncul `0` (hlm. 192).

Secara matematis, jika vocabulary berukuran $|V|$, maka dokumen direpresentasikan sebagai $x\in\{0,1\}^{|V|}$. Kelebihan BoW adalah sederhana; kekurangannya adalah **urutan kata dan konteks negasi tidak tersimpan**.

In [3]:
# Ilustrasi dari vocabulary kecil, terpisah dari data latih.
vocab_kecil = ['cat', 'the', 'dog', 'sat']
indeks_kecil = {kata: i for i, kata in enumerate(vocab_kecil)}

def bag_of_words(teks, kamus):
    vektor = np.zeros(len(kamus), dtype=np.float64)
    for kata in set(tokenisasi(teks)):
        if kata in kamus:
            vektor[kamus[kata]] = 1.0
    return vektor

contoh = bag_of_words('the cat sat', indeks_kecil)
print('Vocabulary kecil:', vocab_kecil)
print('BoW("the cat sat"):', contoh.astype(int))
assert np.array_equal(contoh, np.array([1, 1, 0, 1]))

# Menampilkan kelemahan: dua teks berbeda dapat mempunyai BoW identik.
a = bag_of_words('good not bad', {'good': 0, 'not': 1, 'bad': 2})
b = bag_of_words('bad not good', {'good': 0, 'not': 1, 'bad': 2})
print('Representasi dua urutan kata identik:', np.array_equal(a, b))

Vocabulary kecil: ['cat', 'the', 'dog', 'sat']
BoW("the cat sat"): [1 1 0 1]
Representasi dua urutan kata identik: True


## 5. Embedding layer: memilih baris matriks weight

Untuk vocabulary yang besar, pengalian BoW dengan matriks bobot melakukan banyak perkalian terhadap nol. Bab ini memperkenalkan **embedding layer** sebagai cara efisien untuk mengambil baris bobot kata-kata yang muncul dan menjumlahkannya (hlm. 194–195).

Jika matriks embedding $E\in\mathbb{R}^{|V|\times d}$ dan BoW biner $x$, maka:

$$h_{\mathrm{linear}}=xE=\sum_{i:x_i=1} E_i.$$

Persamaan ini berlaku **jika representasi BoW dan operasi penggabungannya konsisten**, misalnya keduanya biner dan menggunakan penjumlahan. Embedding berdimensi $d$ lebih kecil daripada vocabulary.

In [4]:
E_ilustrasi = np.array([[0.1, 0.2],
                        [0.5, -0.1],
                        [-0.3, 0.4],
                        [0.2, 0.3]])
vector = bag_of_words('the cat sat', indeks_kecil)
melalui_bow = vector @ E_ilustrasi
melalui_embedding = E_ilustrasi[np.flatnonzero(vector)].sum(axis=0)
print('Hasil dari BoW × matriks:', melalui_bow)
print('Hasil penjumlahan embedding:', melalui_embedding)
assert np.allclose(melalui_bow, melalui_embedding)

Hasil dari BoW × matriks: [0.8 0.4]
Hasil penjumlahan embedding: [0.8 0.4]


## 6. Arsitektur neural network untuk klasifikasi sentimen

Sesuai gagasan utama buku (hlm. 194–197), model terdiri atas:

`Indeks kata → Jumlah embedding → Sigmoid hidden layer → Linear layer → Sigmoid → P(positif)`

Notasi:

$$z=\sum_{i\in D}E_i, \qquad h=\sigma(z), \qquad \hat y=\sigma(h^Tw),$$

$$\sigma(t)=\frac{1}{1+e^{-t}},\qquad
L_{\mathrm{BCE}}=-[y\log(\hat y)+(1-y)\log(1-\hat y)].$$

**Binary cross-entropy (BCE)** memberi penalti besar pada prediksi salah yang sangat yakin. Pelatihan menurunkan loss melalui **backpropagation**.

**Catatan reproduksi:** Gagasan lapisan dan tugas mengikuti bab, tetapi kode di bawah ditulis ulang dan menerapkan turunan sigmoid pada hidden layer secara eksplisit menggunakan *chain rule*. Ini merupakan penjelasan gradien yang lebih lengkap dibanding penyederhanaan pada cuplikan kode buku.

In [5]:
def sigmoid(x):
    x = np.clip(x, -30.0, 30.0)  # mencegah overflow numerik
    return 1.0 / (1.0 + np.exp(-x))

def encoding_indeks(teks, kamus):
    # set memastikan setiap kata dihitung sekali, sesuai BoW biner.
    return np.array(sorted({kamus[k] for k in tokenisasi(teks) if k in kamus}),
                    dtype=np.int64)

X_latih = [encoding_indeks(t, word2index) for t in teks_latih]
X_uji = [encoding_indeks(t, word2index) for t in teks_uji]

# Inisialisasi dua matriks/bobot yang akan dipelajari.
hidden_size = 24
model_rng = np.random.default_rng(7)
weights_0_1 = model_rng.uniform(-0.1, 0.1, (len(vocab), hidden_size))
weights_1_2 = model_rng.uniform(-0.1, 0.1, hidden_size)

# Fungsi prediksi tetap berlaku bila semua kata pada ulasan belum dikenal (OOV).
def prediksi_sentimen(indeks, embedding, bobot_output):
    z = embedding[indeks].sum(axis=0)
    h = sigmoid(z)
    prob_positif = float(sigmoid(h @ bobot_output))
    return prob_positif, h

print('Bentuk matriks embedding:', weights_0_1.shape)
print('Bentuk bobot output:', weights_1_2.shape)
print('Probabilitas sebelum belajar:', round(prediksi_sentimen(X_latih[0], weights_0_1, weights_1_2)[0], 4))

Bentuk matriks embedding: (102, 24)
Bentuk bobot output: (24,)
Probabilitas sebelum belajar: 0.5456


## 7. Backpropagation dan pembelajaran model sentimen

Alur pelatihan untuk satu ulasan:

1. **Forward propagation**: menghitung probabilitas positif $\hat y$.
2. **Compare**: menghitung BCE antara prediksi dan label sebenarnya.
3. **Learn**: memperbarui parameter dengan *gradient descent*.

Untuk BCE dengan output sigmoid, delta logit output adalah $\delta_o=\hat y-y$.

$$\frac{\partial L}{\partial w}=h\delta_o,$$

$$\delta_h=(w\delta_o)\odot h\odot(1-h),$$

$$\frac{\partial L}{\partial E_i}=\delta_h \quad \text{untuk setiap kata unik }i\text{ dalam ulasan}.$$

Bobot diperbarui dengan $\theta\leftarrow\theta-\alpha\nabla_\theta L$, dengan $\alpha$ sebagai *learning rate*. Urutan pembaruan penting: turunan dihitung dari bobot **sebelum** diperbarui.

In [6]:
learning_rate = 0.05
epochs = 65 if not USE_IMDB_FILES else 4
riwayat_loss = []

for epoch in range(epochs):
    urutan = model_rng.permutation(len(X_latih))
    total_loss = 0.0
    for j in urutan:
        indeks = X_latih[j]
        y = float(y_latih[j])
        p, hidden = prediksi_sentimen(indeks, weights_0_1, weights_1_2)
        p_aman = np.clip(p, 1e-10, 1 - 1e-10)
        total_loss += -y * np.log(p_aman) - (1 - y) * np.log(1 - p_aman)

        delta_out = p - y
        grad_output = hidden * delta_out
        grad_hidden = weights_1_2 * delta_out * hidden * (1 - hidden)

        weights_1_2 -= learning_rate * grad_output
        if len(indeks):
            weights_0_1[indeks] -= learning_rate * grad_hidden

    rata_loss = total_loss / len(X_latih)
    riwayat_loss.append(float(rata_loss))
    if epoch in (0, epochs // 2, epochs - 1):
        print(f'Epoch {epoch+1:02d}/{epochs} | BCE rata-rata: {rata_loss:.4f}')

print('Loss awal:', round(riwayat_loss[0], 4))
print('Loss akhir:', round(riwayat_loss[-1], 4))

Epoch 01/65 | BCE rata-rata: 0.7173
Epoch 33/65 | BCE rata-rata: 0.4478
Epoch 65/65 | BCE rata-rata: 0.0719
Loss awal: 0.7173
Loss akhir: 0.0719


## 8. Evaluasi prediksi dan batas generalisasi

Akurasi mengukur proporsi prediksi yang benar dengan ambang $0.5$. Kita juga menghitung **baseline kelas mayoritas**. Peningkatan atas baseline di dataset kecil belum cukup untuk membuktikan keberhasilan pada teks di dunia nyata.

**Keterbatasan metodologis:** Data demonstrasi ditulis secara terarah, berukuran kecil, dan memiliki pola leksikal yang mudah. Pemisahan *train/test* membantu tetapi tidak menghilangkan bias tersebut. Untuk penilaian ilmiah, diperlukan dataset eksternal, pemisahan yang terdokumentasi, analisis kesalahan, serta beberapa *random seed*.

In [7]:
def evaluasi_sentimen(X, y, embedding, bobot):
    peluang = np.array([prediksi_sentimen(x, embedding, bobot)[0] for x in X])
    y_pred = (peluang >= 0.5).astype(int)
    akurasi = np.mean(y_pred == y)
    return akurasi, peluang, y_pred

acc_latih, prob_latih, pred_latih = evaluasi_sentimen(X_latih, y_latih, weights_0_1, weights_1_2)
acc_uji, prob_uji, pred_uji = evaluasi_sentimen(X_uji, y_uji, weights_0_1, weights_1_2)
baseline = max(np.mean(y_uji == 0), np.mean(y_uji == 1))

print(f'Akurasi latih: {acc_latih:.3f}')
print(f'Akurasi uji  : {acc_uji:.3f}')
print(f'Baseline uji: {baseline:.3f}')
for teks, benar, proba in list(zip(teks_uji, y_uji, prob_uji))[:5]:
    print(f'P(positif)={proba:.2f} | label={benar} | {teks[:85]}')

Akurasi latih: 1.000
Akurasi uji  : 0.786
Baseline uji: 0.500
P(positif)=0.90 | label=1 | a charming movie with lovely characters
P(positif)=0.34 | label=0 | a tedious movie with irritating characters
P(positif)=0.68 | label=0 | terrible performances from the whole cast
P(positif)=0.79 | label=1 | this film is entertaining and funny
P(positif)=0.49 | label=1 | what a fantastic and thoughtful story


## 9. Interpreting the output: apa yang dipelajari embedding?

Bobot setiap kata di `weights_0_1` adalah sebuah vektor dengan `hidden_size` dimensi. Vektor ini dipelajari agar bermanfaat dalam **memprediksi label sentimen**.

Buku menunjukkan bahwa kata-kata yang memiliki kegunaan serupa untuk prediksi sentimen dapat memperoleh embedding yang berdekatan (hlm. 198–200). Kedekatan ini bersifat **spesifik terhadap tujuan pelatihan**, bukan jaminan kesamaan makna umum.

Salah satu cara mengukur kedekatan adalah **jarak Euclidean**:

$$d(a,b)=\sqrt{\sum_{j=1}^{d}(a_j-b_j)^2}.$$

Nilai jarak yang lebih kecil menunjukkan posisi yang lebih dekat dalam ruang embedding.

In [8]:
def kata_terdekat(kata, embedding, kamus, jumlah=6):
    if kata not in kamus:
        return []
    idx = kamus[kata]
    jarak = np.linalg.norm(embedding - embedding[idx], axis=1)
    urutan = np.argsort(jarak)
    kebalikan = {v: k for k, v in kamus.items()}
    return [(kebalikan[int(i)], round(float(jarak[i]), 4))
            for i in urutan if i != idx][:jumlah]

for kata in ('excellent', 'terrible', 'beautiful'):
    print(kata, '→', kata_terdekat(kata, weights_0_1, word2index, jumlah=6))

excellent → [('lovely', 0.3221), ('delightful', 0.3832), ('entertaining', 0.4142), ('brilliant', 0.4179), ('enjoyed', 0.4502), ('great', 0.4615)]
terrible → [('disappointing', 0.3328), ('confusing', 0.3483), ('awful', 0.369), ('poor', 0.3852), ('horrible', 0.4133), ('bad', 0.4233)]
beautiful → [('inspiring', 0.5482), ('brilliant', 0.7564), ('lovely', 0.7579), ('great', 0.8138), ('excellent', 0.8621), ('amazing', 0.8718)]


## 10. Keterbatasan arsitektur bag of words

Arsitektur berdasarkan **himpunan kata** memiliki kelemahan fundamental: ia mengabaikan urutan kata. Misalnya, dua kalimat `"good not bad"` dan `"bad not good"` memiliki anggota vocabulary yang sama. Karena BoW-nya sama, semua lapisan setelah BoW akan menerima masukan yang sama dan tidak bisa membedakannya.

**Implikasi:** Menambah jumlah neuron tidak dapat memulihkan informasi urutan yang sudah dihilangkan oleh prapemrosesan. Buku menggunakan keterbatasan ini untuk menjelaskan mengapa diperlukan tujuan prediksi dan arsitektur yang lebih sesuai (hlm. 197–198).

In [9]:
kalimat_a = 'good not bad'
kalimat_b = 'bad not good'
indeks_a = encoding_indeks(kalimat_a, word2index)
indeks_b = encoding_indeks(kalimat_b, word2index)
print('Indeks BoW sama:', np.array_equal(indeks_a, indeks_b))
print('Prediksi kalimat A:', round(prediksi_sentimen(indeks_a, weights_0_1, weights_1_2)[0], 5))
print('Prediksi kalimat B:', round(prediksi_sentimen(indeks_b, weights_0_1, weights_1_2)[0], 5))
assert np.array_equal(indeks_a, indeks_b)

Indeks BoW sama: True
Prediksi kalimat A: 0.18611
Prediksi kalimat B: 0.18611


## 11. Fill in the blank: belajar dari konteks kata

Bab ini kemudian mengubah tugas menjadi **memprediksi kata yang dihilangkan dari suatu potongan kalimat** (hlm. 200–203). Misalnya:

`the movie was [____] and moving` → kata yang hilang mungkin `beautiful`.

Implementasi di sini mengikuti gagasan **Continuous Bag of Words (CBOW)**: gunakan embedding beberapa kata di sekitar posisi target untuk memprediksi kata di tengah.

$$c=\frac{1}{m}\sum_{j\in\mathrm{context}} E_j, \qquad
s_k = c^T O_k, \qquad P_k=\sigma(s_k),$$

$E$ adalah matriks embedding konteks, $O$ adalah matriks embedding output; $m$ banyaknya kata konteks. Berbeda dari prediksi sentimen, **target sekarang berupa identitas kata**, bukan label positif/negatif.

Perbedaan yang perlu diingat: representasi konteks mengingat kata-kata **di sekitar** suatu posisi, tetapi merata-ratakan kata konteks sehingga masih tidak memodelkan urutannya secara eksplisit.

In [10]:
# Untuk eksperimen bahasa, setiap kemunculan kata dipertahankan (tidak memakai set).
# Kita tetap menggunakan hanya ulasan latih pada mode demonstrasi.
kalimat_korpus = [tokenisasi(t) for t in teks_latih]

# Vocabulary dibentuk dari kalimat latih, dengan frekuensi untuk negative sampling.
frekuensi = Counter(k for kal in kalimat_korpus for k in kal)
vocab_cbow = sorted(frekuensi)
cbow_word2index = {kata: i for i, kata in enumerate(vocab_cbow)}
cbow_index2word = {i: kata for kata, i in cbow_word2index.items()}
korpus_indeks = [[cbow_word2index[k] for k in kal] for kal in kalimat_korpus]

WINDOW = 2

def konteks_dan_target(kalimat, posisi, radius=WINDOW):
    kiri = kalimat[max(0, posisi-radius):posisi]
    kanan = kalimat[posisi+1:posisi+radius+1]
    return kiri + kanan, kalimat[posisi]

kalimat_demo = tokenisasi('the movie was beautiful and moving')
id_demo = [cbow_word2index[k] for k in kalimat_demo if k in cbow_word2index]
if len(id_demo) >= 3:
    context, target = konteks_dan_target(id_demo, 2)
    print('Konteks:', [cbow_index2word[i] for i in context])
    print('Target :', cbow_index2word[target])
print('Ukuran vocabulary CBOW:', len(vocab_cbow))

Konteks: ['the', 'movie', 'beautiful', 'and']
Target : was
Ukuran vocabulary CBOW: 102


## 12. Negative sampling: memprediksi beberapa kata saja

Jika vocabulary memiliki ribuan kata, menghitung probabilitas penuh seluruh vocabulary pada setiap langkah akan mahal. **Negative sampling** mengubah satu pelatihan menjadi pembedaan antara:
- **1 kata positif**, yaitu kata yang benar-benar ada pada posisi target;
- **K kata negatif**, yaitu kata acak yang digunakan sebagai pembanding.

Kita memberikan target $1$ untuk kata benar dan $0$ untuk kata negatif, lalu meminimalkan BCE untuk sampel-sampel ini.

**Trade-off:** komputasi menjadi lebih hemat karena hanya beberapa baris matriks output yang diperbarui, tetapi hasil probabilitas dari *negative sampling* tidak otomatis merupakan distribusi probabilitas ter-normalisasi atas seluruh vocabulary.

Buku menggunakan teknik ini untuk melatih model prediksi kata (hlm. 201–202). Dalam implementasi ini, sampling disusun agar tidak menyertakan kata positif sebagai contoh negatif pada iterasi yang sama dan pengulangan indeks ditangani dengan `np.add.at`.

In [11]:
# Inisialisasi embedding masukan dan output untuk tugas prediksi kata.
cbow_rng = np.random.default_rng(11)
DIM_EMBED = 20
N_NEGATIF = 5
cbow_lr = 0.04
cbow_epochs = 7 if not USE_IMDB_FILES else 1

input_embeddings = cbow_rng.uniform(-0.1, 0.1, (len(vocab_cbow), DIM_EMBED))
output_embeddings = np.zeros((len(vocab_cbow), DIM_EMBED), dtype=float)

# Distribusi mengikuti frekuensi kemunculan kata; salah satu opsi negative sampling.
frek_array = np.array([frekuensi[k] for k in vocab_cbow], dtype=float)
prob_sampling = frek_array / frek_array.sum()

sample_konteks, sample_target = konteks_dan_target(korpus_indeks[0], 2)

# Satu langkah forward sebelum pelatihan:
c = input_embeddings[sample_konteks].mean(axis=0)
sample_neg_p = prob_sampling.copy()
sample_neg_p[sample_target] = 0
sample_neg_p /= sample_neg_p.sum()
neg = cbow_rng.choice(len(vocab_cbow), size=N_NEGATIF,
                      replace=True, p=sample_neg_p)
id_sampel = np.concatenate([[sample_target], neg])
prob = sigmoid(output_embeddings[id_sampel] @ c)
print('Indeks target dan negatif:', id_sampel.tolist())
print('Probabilitas sebelum belajar:', np.round(prob, 4))
print('Label yang diharapkan:', np.array([1] + [0]*N_NEGATIF))

Indeks target dan negatif: [44, 72, 8, 79, 0, 5]
Probabilitas sebelum belajar: [0.5 0.5 0.5 0.5 0.5 0.5]
Label yang diharapkan: [1 0 0 0 0 0]


## 13. Pelatihan CBOW dengan NumPy

Pada setiap token target, kita menghitung rata-rata embedding konteks, memberi skor untuk satu kata benar dan beberapa kata negatif, kemudian memperbarui embedding konteks maupun embedding output.

Dengan $t_k\in\{0,1\}$ dan $p_k=\sigma(c^TO_k)$:

$$\frac{\partial L}{\partial O_k} = (p_k-t_k)c,$$

$$\frac{\partial L}{\partial c}=\sum_k(p_k-t_k)O_k,$$

$$\frac{\partial L}{\partial E_j}=\frac{1}{m}\frac{\partial L}{\partial c}\quad (j\in\mathrm{context}).$$

**Pemeriksaan penting:** beberapa kata konteks atau kata negatif dapat muncul lebih dari sekali. Karena itu, `np.add.at` dipakai untuk mengakumulasikan semua pembaruan indeks yang berulang secara benar.

In [12]:
riwayat_cbow = []
for epoch in range(cbow_epochs):
    total_loss, langkah = 0.0, 0
    for k in cbow_rng.permutation(len(korpus_indeks)):
        kalimat = korpus_indeks[k]
        for posisi in range(len(kalimat)):
            ctx_ids, pos_id = konteks_dan_target(kalimat, posisi)
            if not ctx_ids:
                continue
            ctx_ids = np.asarray(ctx_ids, dtype=np.int64)
            konteks_vektor = input_embeddings[ctx_ids].mean(axis=0)

            p_neg = prob_sampling.copy()
            p_neg[pos_id] = 0
            p_neg /= p_neg.sum()
            sampel_neg = cbow_rng.choice(len(vocab_cbow), size=N_NEGATIF,
                                         replace=True, p=p_neg)
            sampel = np.concatenate(([pos_id], sampel_neg))
            tujuan = np.zeros(N_NEGATIF + 1)
            tujuan[0] = 1

            skor = output_embeddings[sampel] @ konteks_vektor
            p = sigmoid(skor)
            grad_skor = p - tujuan
            total_loss += float(np.sum(np.logaddexp(0.0, skor) - tujuan * skor))
            langkah += 1

            # Semua gradien menggunakan nilai bobot yang belum diperbarui.
            grad_konteks = grad_skor @ output_embeddings[sampel]
            grad_output = grad_skor[:, None] * konteks_vektor[None, :]

            np.add.at(output_embeddings, sampel, -cbow_lr * grad_output)
            np.add.at(input_embeddings, ctx_ids,
                      -(cbow_lr / len(ctx_ids)) * grad_konteks)

    loss_rata = total_loss / max(langkah * (N_NEGATIF + 1), 1)
    riwayat_cbow.append(loss_rata)
    print(f'Epoch {epoch+1:02d}/{cbow_epochs} | BCE per pasangan: {loss_rata:.4f}')

print('Loss CBOW awal:', round(riwayat_cbow[0], 4))
print('Loss CBOW akhir:', round(riwayat_cbow[-1], 4))

Epoch 01/7 | BCE per pasangan: 0.6926
Epoch 02/7 | BCE per pasangan: 0.6903


Epoch 03/7 | BCE per pasangan: 0.6791


Epoch 04/7 | BCE per pasangan: 0.6467
Epoch 05/7 | BCE per pasangan: 0.5916
Epoch 06/7 | BCE per pasangan: 0.5599
Epoch 07/7 | BCE per pasangan: 0.5341
Loss CBOW awal: 0.6926
Loss CBOW akhir: 0.5341


## 14. Membandingkan embedding dari dua tugas

Kita sekarang memiliki **dua embedding berbeda**:

1. `weights_0_1`: dilatih untuk memprediksi **positif atau negatif**.
2. `input_embeddings`: dilatih untuk **memprediksi kata dari konteks**.

Meskipun teks sumber sama, sasaran optimasinya berbeda sehingga kata-kata yang dianggap “mirip” juga dapat berbeda. Buku memperlihatkan fenomena ini dengan membandingkan hasil pencarian tetangga terdekat dari dua model (hlm. 203–205).

**Perhatian:** karena korpus notebook sangat kecil, tetangga terdekat bisa tampak acak. Ini bukan alasan untuk menyimpulkan suatu metode gagal secara umum.

In [13]:
for target_kata in ('beautiful', 'terrible', 'excellent'):
    print('\nKata:', target_kata)
    print('  Embedding sentimen:', kata_terdekat(target_kata, weights_0_1, word2index, 5))
    print('  Embedding konteks :', kata_terdekat(target_kata, input_embeddings, cbow_word2index, 5))


Kata: beautiful
  Embedding sentimen: [('inspiring', 0.5482), ('brilliant', 0.7564), ('lovely', 0.7579), ('great', 0.8138), ('excellent', 0.8621)]
  Embedding konteks : [('enjoyed', 0.2433), ('awful', 0.2633), ('bad', 0.2641), ('lovely', 0.2747), ('disliked', 0.2871)]

Kata: terrible
  Embedding sentimen: [('disappointing', 0.3328), ('confusing', 0.3483), ('awful', 0.369), ('poor', 0.3852), ('horrible', 0.4133)]
  Embedding konteks : [('beautiful', 0.2959), ('truly', 0.2987), ('horrible', 0.3048), ('bad', 0.3073), ('awful', 0.3094)]

Kata: excellent
  Embedding sentimen: [('lovely', 0.3221), ('delightful', 0.3832), ('entertaining', 0.4142), ('brilliant', 0.4179), ('enjoyed', 0.4502)]
  Embedding konteks : [('noisy', 0.2437), ('made', 0.265), ('slow', 0.2707), ('rich', 0.2824), ('lovely', 0.2924)]


## 15. Makna ditentukan oleh loss function

Salah satu gagasan terpenting dari Chapter 11 adalah bahwa **pola yang dipelajari model bergantung pada masalah optimasi yang diberikan**.

Dua model bisa menerima teks serupa tetapi menghasilkan representasi berbeda karena:
- target yang diprediksi berbeda;
- fungsi kerugian (*loss function*) berbeda;
- arsitektur dan kapasitas model berbeda;
- distribusi data serta prosedur pelatihan berbeda.

Di buku, perubahan dari tugas klasifikasi sentimen ke *fill in the blank* mengubah jenis kedekatan kata yang terbentuk. Hal ini bukan berarti loss “menyimpan semua pengetahuan”, melainkan bahwa **loss memberi sinyal pelatihan** yang mengarahkan bobot untuk menangkap fitur tertentu (hlm. 203–205).

**Implikasi penelitian:** Sebelum mengubah arsitektur, periksa apakah target, data, dan loss memang mengharuskan model mempelajari kemampuan yang diinginkan.

## 16. Word analogy: `king − man + woman ≈ queen`

Vektor kata dapat menunjukkan pola hubungan linear. Jika suatu ruang embedding memisahkan informasi tentang relasi keluarga atau jenis kelamin dan status kerajaan secara konsisten, operasi:

$$v_{\mathrm{king}} - v_{\mathrm{man}} + v_{\mathrm{woman}}\approx v_{\mathrm{queen}}$$

mungkin menghasilkan vektor yang dekat dengan `queen`.

Buku membahas analogi ini sebagai **sifat yang dapat muncul**, bukan identitas aljabar yang selalu benar (hlm. 206–207). Hasilnya bergantung pada korpus, teknik pelatihan, dimensi, definisi jarak, dan bias representasi.

Pertama, kita ilustrasikan prinsip vektornya dengan angka **yang dibuat secara manual**, bukan hasil pelatihan.

In [14]:
# Vektor ilustratif: sengaja dirancang agar analoginya tepat.
vektor_ilustrasi = {
    'man': np.array([0.8, 0.1]),
    'woman': np.array([0.1, 0.9]),
    'king': np.array([0.9, 0.7]),
    'queen': np.array([0.2, 1.5]),
}
pertanyaan_analogi = (vektor_ilustrasi['king']
                     - vektor_ilustrasi['man']
                     + vektor_ilustrasi['woman'])
print('king - man + woman =', np.round(pertanyaan_analogi, 3))
print('queen                =', vektor_ilustrasi['queen'])
print('Jarak ke queen:', np.linalg.norm(pertanyaan_analogi - vektor_ilustrasi['queen']))
assert np.allclose(pertanyaan_analogi, vektor_ilustrasi['queen'])

king - man + woman = [0.2 1.5]
queen                = [0.2 1.5]
Jarak ke queen: 2.7755575615628914e-17


## 17. Menguji analogi pada embedding hasil pelatihan

Berikut fungsi **pencarian kandidat analogi menggunakan embedding hasil pelatihan**, tidak menggunakan vektor ilustrasi. Kode menghitung $v_A-v_B+v_C$, lalu mengurutkan kata terdekat berdasarkan jarak Euclidean sambil mengecualikan kata masukan.

Karena vocabulary IMDB demonstrasi tidak memiliki kata `king`, `man`, `woman`, dan `queen` secara lengkap, kita menguji kombinasi yang tersedia (misalnya `excellent - terrible + bad`). **Tidak ada jaminan** bahwa kata keluaran memiliki hubungan analogi bermakna pada korpus kecil.

In [15]:
def analogy(positif, negatif, embeddings, kamus, top_k=6):
    semua = positif + negatif
    tidak_ada = [kata for kata in semua if kata not in kamus]
    if tidak_ada:
        return {'error': 'Kata tidak tersedia di vocabulary', 'kata': tidak_ada}
    vektor = sum((embeddings[kamus[k]] for k in positif),
                 np.zeros(embeddings.shape[1]))
    vektor -= sum((embeddings[kamus[k]] for k in negatif),
                  np.zeros(embeddings.shape[1]))
    jarak = np.linalg.norm(embeddings - vektor, axis=1)
    terlarang = {kamus[k] for k in semua}
    idx2kata = {i: k for k, i in kamus.items()}
    return [(idx2kata[int(i)], round(float(jarak[i]), 4))
            for i in np.argsort(jarak) if int(i) not in terlarang][:top_k]

print('Analogi embedding konteks (hasil belajar):')
print(analogy(['excellent', 'bad'], ['terrible'], input_embeddings, cbow_word2index))
print('Analogi king - man + woman (jika tidak ada kata akan diberi informasi):')
print(analogy(['king', 'woman'], ['man'], input_embeddings, cbow_word2index))

Analogi embedding konteks (hasil belajar):
[('made', 0.3007), ('noisy', 0.3915), ('me', 0.4206), ('slow', 0.4293), ('confusing', 0.441), ('lovely', 0.4496)]
Analogi king - man + woman (jika tidak ada kata akan diberi informasi):
{'error': 'Kata tidak tersedia di vocabulary', 'kata': ['king', 'woman', 'man']}


## 18. Kesimpulan Chapter 11

1. **NLP** mengolah bahasa menjadi data yang dapat digunakan untuk tugas prediksi.
2. **BoW / one-hot encoding** sederhana dan efektif sebagai dasar, tetapi tidak memperhatikan urutan kata.
3. **Embedding layer** mengganti perhitungan matriks yang banyak bernilai nol dengan pengambilan dan penggabungan baris bobot kata.
4. **Klasifikasi sentimen** mengajari embedding membedakan kata yang membantu memprediksi label positif/negatif.
5. **Fill in the blank (CBOW)** mengajarkan representasi kata berdasarkan konteks lokal.
6. **Negative sampling** mempercepat optimasi dengan membandingkan kata target terhadap beberapa kata negatif.
7. **Loss function dan pilihan target** memengaruhi sifat representasi internal model.
8. **Word analogy** adalah fenomena yang mungkin muncul pada embedding yang terlatih baik, bukan sifat universal setiap model.